# 114 · 1 · Embeddings: input, latent, penultimate
Three views of the same samples (`df`, train + eval with a `split` column), each reduced to 3D (`REDUCER` = `pca` or `tsne`), side by side.
One dropdown recolours all three panels together.

| panel | what it is | where it is grabbed (forward hook) |
|---|---|---|
| **input** | the per-box standardized spectrum, `(B, 100 lasers, 1170 bins, 2 axes)` → 234,000 | input of `model.encoder.freq_enc` |
| **latent** | the encoder output, `(B, 1024)`: what the decoder (and every aux head) reads | input of `model.decoder` |
| **penultimate** | the last decoder feature map, `(B, 32, 64, 64)` → 131,072 | input of `model.decoder.conv3` (the final 32→2 conv: mask + outline logits) |

**Where normalization happens.** `Transformer.forward()` receives the raw `batch["fft"]` (`|fft| / its std over the whole sample`, built by
`dataset.process_x`). In eval mode the train-time augs (speaker gain, laser jitter) are off, and `Encoder.forward` calls
`Encoder.normalize(x, box)` = `(x - cum_mean[box]) * cum_std[box]`: per-box, per-(laser, bin, axis) standardization with the train-set stats
stored as buffers in the checkpoint. Its output goes straight into `encoder.freq_enc`, so the hook on `freq_enc`'s input sees exactly the normalized tensor.

**No CLS token.** The laser transformer has no CLS / pooled token: its 100 laser tokens (128-d each) go through `LayerNorm → Flatten → Linear(12800, 1024)`
(`LaserEncoder.to_emb`). That 1024-d vector is the only thing the decoder sees, so it is the latent.

**Reduction.** PCA is exact, through the centred `N×N` Gram matrix (features stay on the GPU in fp16, no `D×D` covariance). t-SNE / UMAP run on the top 50 PCs, fit on all points (train + eval) alike.
**speckles** = number of connected components (`scipy.ndimage.label`, 4-connectivity) of the predicted mask at `THR = 0.08` (the codebase has no other definition;
"unseen speckles" in nb113 means the laser-shifted splits, unrelated). **x / y** = the mean CoM (col / row) of the objects, box fractions (row 0 = laser wall side);
empty-box samples have no position (gray). **area** = target mask area, px of 64×64.

# 114 setup: run, split, data, model (shared by every section)
`RUN` picks a src9 run; the split is the one src9 trains on (`src9/dataset.load_splits`). Data = a fixed random subset (`N_TRAIN` train samples +
up to `N_PER_EVAL` per eval split), spectra computed exactly like `dataset.process_x` (tukey, |fft| / std) and cached in `runs/_nb114_cache/<RUN>/`
(fp16 memmap, ~1.4 GB on disk, read lazily: CPU stays small). `df` has one row per sample, with a `split` column (`train` or `eval/<box>/<name>`).

The checkpoint was saved by an earlier version of `arch/transformer.py` (flat names `encoder.embed`, `encoder.lasers`, ...); the working tree groups
them into `encoder.freq_enc` / `encoder.laser_enc`. Same weights, so the keys are renamed on load. The per-box normalization stats (`cum_mean`,
`cum_std`) are buffers in the checkpoint.

In [1]:
import os, sys, json, math
os.environ["TQDM_DISABLE"] = "1"
from pathlib import Path
import numpy as np, pandas as pd, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src9").exists())
sys.path.insert(0, str(ROOT / "src9")); import dataset as D9
from arch.transformer import Transformer

RUN = "s9-transformer-vase-1k"
CKPT = "latest-rank0.pt"
METRIC = "iou@0.08"
REDUCER = "pca"            # pca | tsne | umap
N_TRAIN, N_PER_EVAL, SEED = 1000, 40, 0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
args = json.loads((ROOT / f"runs/{RUN}/args.json").read_text())
assert args["model"] == "transformer" and args["window"] == "tukey", args
CACHE = ROOT / f"runs/_nb114_cache/{RUN}"; CACHE.mkdir(parents=True, exist_ok=True)
print(RUN, args["max_duration"], DEV, torch.cuda.mem_get_info() if DEV == "cuda" else "")

s9-transformer-vase-1k 1000ep cuda (5072093184, 16608133120)


In [2]:
# the split src9 trained on -> a fixed subset -> df (one row per sample)
samples, idx = D9.load_splits()
metas, dirs = list(samples.values()), list(samples)
rng = np.random.default_rng(SEED)
pick = {int(i): "train" for i in rng.choice(idx["train"], N_TRAIN, replace=False)}
for name, ids in idx.items():
    if name == "train": continue
    for i in rng.choice(ids, min(N_PER_EVAL, len(ids)), replace=False): pick.setdefault(int(i), name)
order = sorted(pick)
boxes = sorted({m["box"] for m in metas})
objects = sorted({"+".join(sorted(metas[i]["objects"])) for i in idx["train"]}) + ["unseen"]   # as build_dataset: sets the object head size
df = pd.DataFrame([dict(i=i, split=pick[i], box=metas[i]["box"],
                        object="+".join(sorted(metas[i]["objects"])) or "empty", n_objects=metas[i]["n_objects"], speaker=metas[i]["speaker"],
                        experiment=metas[i]["experiment"], position_id=metas[i]["position_id"], sample_id=metas[i]["sample_id"], layout=metas[i]["layout"])
                   for i in order])
df["is_train"] = df.split == "train"
n_objects_all, max_k = sorted({m["n_objects"] for m in metas}), max(m["n_objects"] for m in metas)
sub = {dirs[i]: metas[i] for i in order}                  # only the picked samples' metadata; the rest (~0.8 GB of dicts) is dropped
del samples, metas, dirs; import gc; gc.collect()
print(df.split.value_counts().head(), len(df), "samples,", df.split.nunique(), "splits")

2026_09_27_gastronorm_four_objs: 6155 samples


2026_10_05_green_plastic_four_objs: 5736 samples


2026_10_11_cardboard_four_objs: 7228 samples


split                                          samples    positions
-------------------------------------------  ---------  -----------
train                                            11219         3032
eval/gastro/vase                                   135           45
eval/gastro/candle                                 156           52
eval/gastro/cube                                   147           49
eval/gastro/cube-unseen-speaker-baseline           105           35
eval/gastro/cube-unseen-speaker-unseen              35           35
eval/gastro/soap-dispenser                         146           49
eval/gastro/cube-vase                               27            9
eval/gastro/two-cubes                               42           14
eval/gastro/cube-with-lid                           15            5
eval/gastro/cube-without-lid                        15            5
eval/gastro/cylinder-1000g                          27            9
eval/gastro/cylinder-500g                       

split
train                                       1000
eval/gastro/cube-unseen-speaker-baseline      40
eval/gastro/cube                              40
eval/gastro/vase                              40
eval/gastro/unseen-position                   40
Name: count, dtype: int64 3178 samples, 66 splits


In [3]:
# spectra X (N,L,F,C) fp16 memmap, targets Y (N,64,64), coms (N,K,2); built once per (RUN, N_TRAIN, N_PER_EVAL, SEED)
tag = f"{N_TRAIN}_{N_PER_EVAL}_{SEED}"
fx, fy, fc = CACHE / f"X_{tag}.npy", CACHE / f"Y_{tag}.npy", CACHE / f"coms_{tag}.npy"
if not fc.exists():
    L, F_, C = D9.load_fft(*next(iter(sub.items())), "hann").shape
    X = np.lib.format.open_memmap(fx, "w+", np.float16, (len(order), L, F_, C))
    for j, x in enumerate(D9.process_x(sub, "tukey")): X[j] = x
    X.flush(); del X
    ys = D9.process_y(sub, objects, boxes, max_k, 64, 64)
    np.save(fy, np.stack([y["y"] for y in ys])); np.save(fc, np.stack([y["coms"] for y in ys]))
X = np.load(fx, mmap_mode="r")                     # (N,100,1170,2) fp16, not normalized (the model normalizes)
Y = np.load(fy); COMS = np.load(fc)
BOX = torch.tensor([boxes.index(b) for b in df.box])
df["area"] = (Y > 0.5).sum((1, 2))                                     # true mask area, px of 64x64
import warnings; warnings.filterwarnings("ignore", "Mean of empty slice")
c = np.where(df.n_objects.values[:, None, None] > np.arange(COMS.shape[1])[None, :, None], COMS, np.nan)
df["com_row"], df["com_col"] = np.nanmean(c[..., 0], 1), np.nanmean(c[..., 1], 1)   # mean CoM of the objects, box fractions (row 1 = laser wall)
print(X.shape, Y.shape)

(3178, 100, 1170, 2) (3178, 64, 64)


In [4]:
# model: the working-tree arch, the checkpoint's weights (old flat names -> freq_enc / laser_enc)
sd = torch.load(ROOT / f"runs/{RUN}/checkpoints/{CKPT}", map_location="cpu", weights_only=False)["state"]["model"]
REN = {"embed": "freq_enc.embed", "band_emb": "freq_enc.freq_band_emb", "box_emb": "freq_enc.box_emb", "freqs": "freq_enc.freqs", "pool_q": "freq_enc.pool_q",
       "pool_norm": "freq_enc.pool_norm", "pool_k": "freq_enc.pool_k", "pool_o": "freq_enc.pool_o", "lasers": "laser_enc.lasers", "to_emb": "laser_enc.to_emb"}
def ren(k):
    p = k.split(".")
    return ".".join([p[0], REN[p[1]], *p[2:]]) if p[0] == "encoder" and p[1] in REN else k
sd = {ren(k): v for k, v in sd.items()}
m0 = next(iter(sub.values()))
info = dict(n_laser_rows=m0["n_rows"], n_laser_cols=m0["n_cols"], n_freqs=X.shape[2], n_channels=X.shape[3], freq_range=(float(m0["min_freq"]), float(m0["max_freq"])),
            objects=objects, n_objects=n_objects_all, boxes=boxes,
            cum_mean=sd["encoder.cum_mean"], cum_var=sd["encoder.cum_std"].float() ** -2)
model = Transformer(info)
print(model.load_state_dict(sd, strict=True))
model = model.to(DEV).eval(); del sd
R, Cc = info["n_laser_rows"], info["n_laser_cols"]; L = R * Cc                      # 10 x 10 laser grid, laser l = (l // Cc, l % Cc)
NB = len(model.encoder.freq_band_logf); BPB = model.encoder.freq_enc.bins_per_band  # 19 freq bands of 64 bins
FREQS = np.linspace(*info["freq_range"], X.shape[2])
BAND_HZ = np.array([FREQS[min(b * BPB + BPB // 2, len(FREQS) - 1)] for b in range(NB)])   # band centres, Hz
xb = lambda ids: torch.from_numpy(X[ids].astype(np.float32)).to(DEV)               # raw (B,L,F,C) on the GPU
print(f"L={L} ({R}x{Cc}), {NB} bands of {BPB} bins ~{FREQS[BPB] - FREQS[0]:.0f} Hz")

<All keys matched successfully>
L=100 (10x10), 19 bands of 64 bins ~49 Hz


In [5]:
# per-sample metrics, (B,H,W) probabilities vs (B,H,W) truth -> (B,), nan for an empty box (as src9/utils.py iou08 / miss08)
THR = 0.08
from utils import moments
nan = lambda v, y: torch.where((y > 0.5).flatten(1).any(1), v, torch.nan)          # empty boxes don't score
def iou_at(t): return lambda p, y: nan(((p > t) & (y > 0.5)).flatten(1).sum(1) / ((p > t) | (y > 0.5)).flatten(1).sum(1).clamp_min(1), y)
def miss_at(t): return lambda p, y: nan((~((p > t) & (y > 0.5)).flatten(1).any(1)).float(), y)
METRICS = {"iou@0.08": iou_at(THR), "miss@0.08": miss_at(THR), "iou@0.5": iou_at(0.5),
           "soft_iou": lambda p, y: nan(torch.minimum(p, y).flatten(1).sum(1) / torch.maximum(p, y).flatten(1).sum(1).clamp_min(1e-6), y),
           "com_err": lambda p, y: nan((moments(p)[1] - moments(y)[1]).norm(dim=-1), y)}   # centroid distance, box fractions
HIGHER_BETTER = {"iou@0.08": True, "miss@0.08": False, "iou@0.5": True, "soft_iou": True, "com_err": False}
metric = METRICS[METRIC]

In [6]:
# baseline forward over everything (bf16 autocast as in training); check against the run's own saved eval scores
@torch.no_grad()
def predict(ids, bs=64):
    out = []
    for s in range(0, len(ids), bs):
        j = ids[s:s + bs]
        with torch.autocast("cuda", torch.bfloat16, enabled=DEV == "cuda"): out.append(model({"fft": xb(j), "box": BOX[j].to(DEV)})["mask_pred"].float())
    return torch.cat(out)
P = predict(np.arange(len(df)))                                                  # (N,64,64) on the GPU
YT = torch.from_numpy(Y).to(DEV)
for k, f in METRICS.items(): df[k] = f(P, YT).cpu().numpy()
print(df.groupby(df.is_train)[list(METRICS)].mean().round(3))
import score
ev = df[~df.is_train]; one = ev.groupby("split")[METRIC].mean()
ref = score.per_sample(RUN, score.last_epoch(RUN), THR).assign(split=lambda d: "eval/" + d.box + "/" + d.split).groupby("split").iou.mean()
j = one.dropna().index.intersection(ref.index)
print(f"{METRIC} here vs run's saved eval (same splits, subset): {one[j].mean():.3f} vs {ref[j].mean():.3f}, per-split corr {np.corrcoef(one[j], ref[j])[0, 1]:.2f}")

          iou@0.08  miss@0.08  iou@0.5  soft_iou  com_err
is_train                                                 
False        0.464      0.139    0.458     0.458    0.072
True         0.911      0.004    0.977     0.911    0.004


iou@0.08 here vs run's saved eval (same splits, subset): 0.441 vs 0.441, per-split corr 1.00


## Grab the three representations

In [7]:
# forward hooks: the input of encoder.freq_enc (normalized spectrum), of the decoder (latent), of decoder.conv3 (penultimate). Kept on the GPU in fp16
feats, SHAPE = {"input": [], "latent": [], "penultimate": []}, {}
def grab(k):
    def hook(mod, inp, out):
        SHAPE[k] = tuple(inp[0].shape[1:]); feats[k].append(inp[0].flatten(1).half())
    return hook
hooks = [model.encoder.freq_enc.register_forward_hook(grab("input")), model.decoder.register_forward_hook(grab("latent")),
         model.decoder.conv3.register_forward_hook(grab("penultimate"))]
P2 = predict(np.arange(len(df)), bs=32)
for h in hooks: h.remove()
feats = {k: torch.cat(v) for k, v in feats.items()}
print(f"max |P2 - P| vs the setup forward: {(P2 - P).abs().max():.3f} (bf16, batch 32 vs 64)")
for k, v in feats.items(): print(f"{k:12s} per sample {SHAPE[k]} -> {tuple(v.shape)}  ({v.numel() * 2 / 1e9:.2f} GB fp16 on GPU)")
tr = torch.tensor(df.is_train.values, device=DEV)
z = feats["input"][tr].float()
print(f"normalized input on the train subset: mean {z.mean():.3f}, std {z.std():.3f} (per-box train stats -> ~0 / ~1)"); del z

max |P2 - P| vs the setup forward: 0.090 (bf16, batch 32 vs 64)
input        per sample (100, 1170, 2) -> (3178, 234000)  (1.49 GB fp16 on GPU)
latent       per sample (1024,) -> (3178, 1024)  (0.01 GB fp16 on GPU)
penultimate  per sample (32, 64, 64) -> (3178, 131072)  (0.83 GB fp16 on GPU)
normalized input on the train subset: mean -0.004, std 0.994 (per-box train stats -> ~0 / ~1)


In [8]:
# colour columns: speckles from the setup predictions P, position from the target CoMs
from scipy import ndimage
df["speckles"] = [ndimage.label(p > THR)[1] for p in P.cpu().numpy()]   # 4-connectivity
df["x"], df["y"] = df.com_col, df.com_row
df["train/eval"] = np.where(df.is_train, "train", "eval")
print(df.speckles.value_counts().sort_index().to_dict())
print(df.groupby("is_train").speckles.mean().round(2).to_dict())

{0: 90, 1: 2745, 2: 332, 3: 11}
{False: 1.08, True: 1.08}


## Reduce to 3D

In [9]:
@torch.no_grad()
def pca(Z, k=50):
    """(N,D) fp16 on the GPU -> (N,k) PCA scores and the explained-variance ratios, exact, via the centred (N,N) Gram matrix"""
    G = torch.zeros(len(Z), len(Z), device=Z.device, dtype=torch.float64)
    for s in range(0, Z.shape[1], 8192):
        z = Z[:, s:s + 8192].float(); z -= z.mean(0); G += (z @ z.T).double()
    w, v = torch.linalg.eigh(G.cpu())                                         # (N,N) is small; CPU eigh (cusolver fails next to training)
    w, v = w.flip(0).clamp_min(0), v.flip(1)
    return (v[:, :k] * w[:k].sqrt()).float().numpy(), (w / w.sum()).numpy()

from sklearn.manifold import TSNE
EMB, EVR = {}, {}
for k, Z in feats.items():
    s, EVR[k] = pca(Z)
    EMB[k] = {"pca": lambda: s[:, :3], "tsne": lambda: TSNE(3, init="pca", perplexity=30, random_state=0).fit_transform(s),
              "umap": lambda: __import__("umap").UMAP(n_components=3, random_state=0).fit_transform(s)}[REDUCER]()
    print(f"{k:12s} PC1-3 explain {EVR[k][:3].round(3)} (sum {EVR[k][:3].sum():.2f}), 50 PCs {EVR[k][:50].sum():.2f}")
del feats; torch.cuda.empty_cache()

input        PC1-3 explain [0.207 0.101 0.067] (sum 0.37), 50 PCs 0.77


latent       PC1-3 explain [0.093 0.053 0.05 ] (sum 0.20), 50 PCs 0.79


penultimate  PC1-3 explain [0.067 0.05  0.036] (sum 0.15), 50 PCs 0.76


## Plot: three panels, one dropdown

In [10]:
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

CAT8 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed order; a 9th level folds into "other"
GRAY = "#a3a29b"
SEQ = [[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]]                                            # one hue, light -> dark
COLOR_BY = ["object", "n_objects", "speaker", "area", "x", "y", "box", "speckles", "train/eval"]
CONT = {"area", "x", "y"}

def colors(col):
    """-> (N,) hex per sample, legend [(label, hex)], (vmin, vmax) or None"""
    if col in CONT:
        v = df[col].astype(float).values; lo, hi = np.nanmin(v), np.nanmax(v)
        c = np.array([GRAY] * len(v), dtype=object); ok = ~np.isnan(v)
        c[ok] = sample_colorscale(SEQ, list((v[ok] - lo) / (hi - lo)))
        return c, ([("n/a (empty box)", GRAY)] if (~ok).any() else []), (lo, hi)
    lab = df[col].clip(upper=6).astype(str).replace("6", "6+") if col == "speckles" else df[col].astype(str)
    lv = list(lab.value_counts().index) if col == "object" else sorted(lab.unique(), key=lambda s: (s.rstrip("+").isdigit() and float(s.rstrip("+")), s))
    if len(lv) > 8: lab = lab.where(lab.isin(lv[:7]), "other"); lv = lv[:7] + ["other"]
    cmap = {l: (GRAY if l == "other" else CAT8[i]) for i, l in enumerate(lv)}
    return lab.map(cmap).values, [(f"{l} ({(lab == l).sum()})", cmap[l]) for l in lv], None

NAMES = ["input", "latent", "penultimate"]
ax = lambda k: f"{'PC' if REDUCER == 'pca' else 'tSNE'}"
titles = [f"<b>{k}</b> {SHAPE[k]} · " + (f"PC1-3 {100 * EVR[k][:3].sum():.0f}% var" if REDUCER == "pca" else f"{REDUCER} of 50 PCs") for k in NAMES]
fig = make_subplots(1, 3, specs=[[{"type": "scene"}] * 3], subplot_titles=titles, horizontal_spacing=0.01)
hover = [f"{r.sample_id} · {r.experiment[:10]}<br>{r.object} (n={r.n_objects}) · spk {r.speaker}<br>pos {r.position_id} (row {r.com_row:.2f}, col {r.com_col:.2f})"
         f"<br>{r.split} · area {r.area} · speckles {r.speckles}" for r in df.itertuples()]
C = {col: colors(col) for col in COLOR_BY}
for i, k in enumerate(NAMES):                                                      # traces 0-2: the data, recoloured by the dropdown
    e = EMB[k]
    fig.add_trace(go.Scatter3d(x=e[:, 0], y=e[:, 1], z=e[:, 2], mode="markers", marker=dict(size=2.5, color=C[COLOR_BY[0]][0], opacity=0.85, line_width=0),
                               hovertext=hover, hovertemplate="%{hovertext}<extra></extra>", showlegend=False), 1, i + 1)
owner = []                                                                         # legend-only traces (and one colourbar per continuous column)
for col in COLOR_BY:
    c, leg, rng_ = C[col]
    for name, h in leg:
        fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers", marker=dict(size=8, color=h), name=name, visible=col == COLOR_BY[0]), 1, 1); owner.append(col)
    if rng_:
        fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers", showlegend=False, visible=col == COLOR_BY[0],
                                   marker=dict(color=list(rng_), cmin=rng_[0], cmax=rng_[1], colorscale=SEQ, showscale=True,
                                               colorbar=dict(title=col, len=0.6, thickness=12, x=1.0))), 1, 1); owner.append(col)
leg_color = [t.marker.color for t in fig.data[3:]]
buttons = [dict(label=col, method="update", args=[{"marker.color": [C[col][0]] * 3 + leg_color, "visible": [True] * 3 + [o == col for o in owner]},
                                                  {"legend.title.text": col}]) for col in COLOR_BY]
for i in range(3):
    fig.layout[f"scene{i + 1 if i else ''}"].update(xaxis_title=f"{ax(0)}1", yaxis_title=f"{ax(0)}2", zaxis_title=f"{ax(0)}3", aspectmode="cube")
fig.update_layout(width=2000, height=720, margin=dict(l=10, r=10, t=90, b=10), legend=dict(title=COLOR_BY[0], itemsizing="constant", x=1.0, y=0.98),
                  updatemenus=[dict(buttons=buttons, direction="down", x=0.0, y=1.13, xanchor="left", showactive=True)],
                  title=dict(text=f"{RUN} · {len(df)} samples ({df.is_train.sum()} train) · {REDUCER} · colour by:", x=0.08, y=0.985))
out = ROOT / f"notebooks/out/114_embeddings_{RUN}_{REDUCER}.html"; out.parent.mkdir(exist_ok=True)
fig.write_html(out, include_plotlyjs="cdn"); print(out, f"{out.stat().st_size / 1e6:.1f} MB")
fig.show()

/home/ethantu/workspace/good-vibrations/notebooks/out/114_embeddings_s9-transformer-vase-1k_pca.html 3.1 MB


In [11]:
# what each panel is organized by, in its 3D coords. categorical: share of the 10 nearest neighbours with the same label (chance = sum p^2);
# continuous: R^2 of a linear fit from the 3 coords (rows with a value only)
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
rows = {}
for k in NAMES:
    e = EMB[k]; nn = NearestNeighbors(n_neighbors=11).fit(e).kneighbors(e, return_distance=False)[:, 1:]
    r = {}
    for col in COLOR_BY:
        v = df[col]
        if col in CONT: ok = v.notna().values; r[col] = LinearRegression().fit(e[ok], v[ok]).score(e[ok], v[ok])
        else: lab = np.array(v.astype(str).tolist()); r[col] = (lab[nn] == lab[:, None]).mean()
    rows[k] = r
chance = pd.Series({col: (df[col].astype(str).value_counts(normalize=True) ** 2).sum() for col in COLOR_BY if col not in CONT})
print(pd.DataFrame(rows).assign(chance=chance).round(2))

            input  latent  penultimate  chance
object       0.24    0.22         0.27    0.13
n_objects    0.83    0.83         0.88    0.81
speaker      0.91    0.27         0.25    0.24
area         0.00    0.00         0.02     NaN
x            0.00    0.70         0.63     NaN
y            0.00    0.46         0.09     NaN
box          0.93    0.76         0.52    0.36
speckles     0.79    0.79         0.83    0.76
train/eval   0.60    0.62         0.61    0.57


## Notes (3178 samples: 1000 train + up to 40 per eval split; ep1000)
- **input** (per-box standardized, mean -0.004 / std 0.99 on train): PC1-3 hold 37% of the variance and are organized by **speaker** (10-NN agreement .91, chance .24) and **box** (.93).
  No position at all (R² of x / y / area from PC1-3 = 0.00). The dominant variance in what the model sees is the speaker and the box, not the object.
- **latent** (1024-d): the top PCs turn into **position**: x (column) R² .70, y (row) .46; speaker drops to chance (.27) along PC1-3 and box to .76.
  But t-SNE (`REDUCER="tsne"`, same 50 PCs) still finds speaker (.75) and box (1.00) neighbourhoods. So the latent is not speaker-invariant. Speaker is only pushed off the top PCs.
- **penultimate** (32×64×64 decoder map): speaker is at chance in both PCA and t-SNE (.25 / .24), and box falls to .52 / .68. x stays readable (R² .63), and y is weaker in 3D (.09).
  This is the first representation that is about the scene and not the recording.
- **object** identity is weak in 3D everywhere (PCA .22-.27, t-SNE .43-.45 vs chance .13). **area** is not linearly readable from any 3 coordinates.
- **speckles** (4-connectivity, thr .08): 2745/3178 predictions are a single blob, 332 have 2, 11 have 3, and 90 are empty. Train and eval both average 1.08 blobs, which matches the
  "no speckle" memory note.
- train vs eval sit close together in every panel (10-NN agreement .60-.62 vs chance .57). Eval points are not off-manifold in these coordinates.
- The hook forward runs at batch 32 and the setup at batch 64. Under bf16 the max per-pixel |Δp| between the two is .09, most likely from batch-size-dependent bf16 kernels (not checked). The setup's IoU check against the saved eval stands.